In [3]:
import torch
import torch.nn as nn
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
from GPT_Model import GPT, generate_text, cfg

from Training_pipeline import generate_and_print_sample, text_to_token_id, token_id_to_text

In [4]:
model=GPT(cfg)
model.eval()
print("Model is in Evaluation Mode")

Model is in Evaluation Mode


In [13]:
#prev implemention to find next token generation
tokenid=generate_text(
  model=model,
  ip_token_id=text_to_token_id("Every step moves you",tokenizer),
  max_new_tokens=6,
  context_size=int(cfg["context_len"]/4)
)
print(token_id_to_text(tokenid,tokenizer))

Every step moves you alloy Mo interior� trophy Blessing


## TOP-K Scaling

In [5]:
logits = torch.tensor([-0.0522,  0.7182, 2.4848, 0.0423, 2.3756, -0.0154,  0.2809, 2.3872, 0.3145])

In [6]:
topk=3
top_logit,top_pos=torch.topk(logits,topk)
print(f"top_logit is:{top_logit},\ntop pos:{top_pos}")

top_logit is:tensor([2.4848, 2.3872, 2.3756]),
top pos:tensor([2, 7, 4])


In [7]:
#expact top k every other is set to inf
new_logits =torch.where(
  condition=logits<top_logit[-1],
  input=torch.tensor(float("-inf")),
  other=logits
)
new_logits

tensor([  -inf,   -inf, 2.4848,   -inf, 2.3756,   -inf,   -inf, 2.3872,   -inf])

In [17]:
temp=0.1
new_logits=new_logits/temp
new_logits

tensor([   -inf,    -inf, 24.8480,    -inf, 23.7560,    -inf,    -inf, 23.8720,
           -inf])

## function with Temp+ top_k_scaling

In [ ]:
def generate_text_simple_with_temp_and_topk(model, ip_token_id, max_new_tokens, context_size,temp,topk=None):
  for i in range(max_new_tokens):
    context_ip=ip_token_id[:,-context_size:]
    with torch.no_grad():
      logit=model(context_ip)
    logit=logit[:,-1,:] #tensor size is (batch) x 1 x (vocab_size)  
    
    if temp>0:
      logit=logit/temp
    
    if topk is not None:
      top_logit,top_pos=torch.topk(logits,topk)
      logit=torch.where(
          condition=logit<top_logit[-1],
        input=torch.tensor(float("-inf")),
          other=logit
      )
    
    prob=torch.softmax(logit,dim=-1)#convert to prob , we can take max of logit also that give same but prob give explainibilty
    idx_next=torch.argmax(prob,dim=-1,keepdim=True) #taking max val prob
    ip_token_id=torch.concat((ip_token_id,idx_next),dim=1)
  return ip_token_id  

In [ ]:
#temp+topk implemention to find next token generation
tokenid=generate_text_simple_with_temp_and_topk(
  model=model,
  ip_token_id=text_to_token_id("Every step moves you",tokenizer),
  max_new_tokens=6,
  context_size=int(cfg["context_len"]/4),
  temp=0.1,
  topk=3
)
print(token_id_to_text(tokenid,tokenizer))

Every step moves you alloy Mo interior� trophy Blessing


In [25]:
tokenid=generate_text_simple_with_temp_and_topk(
  model=model,
  ip_token_id=text_to_token_id("Every step moves you",tokenizer),
  max_new_tokens=6,
  context_size=int(cfg["context_len"]/4),
  temp=0.1,
  topk=5
)
print(token_id_to_text(tokenid,tokenizer))

Every step moves you alloy Mo interior� trophy Blessing
